In [1]:
# =============================================================================
# 01_Silver_Cleaning.py — SA Retail Analytics
# Silver Layer: Data Cleaning & Transformation
#
# Purpose: Read raw Bronze CSVs → clean → join → write Silver Delta tables
# Attach Lakehouse: Sales_Storage
# =============================================================================

StatementMeta(, 831c6802-8da8-403f-a5e5-7569e6289849, 3, Finished, Available, Finished, False)

In [2]:
# Imports
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, DataType, StringType

print("Libraries Loaded!")

StatementMeta(, 831c6802-8da8-403f-a5e5-7569e6289849, 4, Finished, Available, Finished, False)

Libraries Loaded!


In [3]:
# Config
BRONZE_PATH = "Files/bronze"
SILVER_DB = "Sales_Storage"   # This is the Lakehouse name, which is our database in Spark

StatementMeta(, 831c6802-8da8-403f-a5e5-7569e6289849, 5, Finished, Available, Finished, False)

In [4]:
# Read Bronze CSVs 
df_customers = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(f"{BRONZE_PATH}/Customer.csv")
)

df_products = (
    spark.read
    .option("header", "true")
    .option("inferScheme", "false")
    .csv(f"{BRONZE_PATH}/Product.csv")
)

df_orders = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(f"{BRONZE_PATH}/Orders.csv")
)

print(f"Customers: {df_customers.count()} rows | {len(df_customers.columns)} cols")
print(f"Products: {df_products.count()} rows | {len(df_products.columns)} cols")
print(f"Orders: {df_orders.count()} rows | {len(df_orders.columns)} cols")

StatementMeta(, 831c6802-8da8-403f-a5e5-7569e6289849, 6, Finished, Available, Finished, False)

Customers: 20 rows | 7 cols
Products: 20 rows | 5 cols
Orders: 352 rows | 12 cols


In [5]:
# Clean Customers
silver_customers = (
    df_customers
    #drop duplicates
    .drop_duplicates(["customer_id"])
    #trim white spaces
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("email", F.trim(F.lower(F.col("email"))))
    .withColumn("phone", F.trim(F.col("phone")))
    .withColumn("city", F.trim(F.initcap(F.col("city"))))
    .withColumn("Province", F.trim(F.col("province")))
    #standardise segment to uppercase
    .withColumn("segment", F.upper(F.trim(F.col("segment"))))
    #drop nulls on fields
    .dropna(subset=["customer_id", "name", "email"])
    #select final columns in order
    .select("customer_id", "name", "email", "phone", "city", "province", "segment")
)

print(f"\nSilver_customers: {silver_customers.count()} rows")
silver_customers.show(5, truncate=False)

StatementMeta(, 831c6802-8da8-403f-a5e5-7569e6289849, 7, Finished, Available, Finished, False)


Silver_customers: 20 rows
+-----------+--------------+------------------------+------------+------------+-------------+-------+
|customer_id|name          |email                   |phone       |city        |province     |segment|
+-----------+--------------+------------------------+------------+------------+-------------+-------+
|C001       |Sipho Dlamini |sipho.dlamini@gmail.com |071 234 5678|Johannesburg|Gauteng      |PREMIUM|
|C002       |Naledi Mokoena|naledi.mokoena@gmail.com|082 345 6789|Soweto      |Gauteng      |REGULAR|
|C003       |Thabo Nkosi   |thabo.nkosi@outlook.com |083 456 7890|Cape Town   |Western Cape |PREMIUM|
|C004       |Zanele Khumalo|zanele.khumalo@yahoo.com|074 567 8901|Durban      |KwaZulu-Natal|BUDGET |
|C005       |Lerato Sithole|lerato.sithole@gmail.com|060 678 9012|Pretoria    |Gauteng      |REGULAR|
+-----------+--------------+------------------------+------------+------------+-------------+-------+
only showing top 5 rows



In [6]:
# Clean Products
silver_products = (
    df_products
    .drop_duplicates(["product_id"])
    .withColumn("name", F.trim(F.col("name")))
    #rename "name" to "product_name" to avoid ambiguity in joins
    .withColumnRenamed("name", "product_name")
    .withColumn("category", F.trim(F.initcap(F.col("category"))))
    #cast numeric types
    .withColumn("price", F.col("price").cast(DoubleType()))
    .withColumn("stock", F.col("stock").cast(IntegerType()))
    #validate price > 0
    .filter(F.col("price") > 0)
    .dropna(subset=["product_id", "product_name"])
    .select("product_id", "product_name", "category", "price", "stock")
)

print(f"\nSilver_Products: {silver_products.count()} rows")
silver_products.show(5, truncate=False)

StatementMeta(, 831c6802-8da8-403f-a5e5-7569e6289849, 8, Finished, Available, Finished, False)


Silver_Products: 20 rows
+----------+---------------------------+---------+-----+-----+
|product_id|product_name               |category |price|stock|
+----------+---------------------------+---------+-----+-----+
|P001      |Sasko White Bread 700g     |Groceries|19.99|500  |
|P002      |Albany Superior Brown Bread|Groceries|21.49|420  |
|P003      |Clover Full Cream Milk 2L  |Groceries|34.99|350  |
|P004      |Koo Baked Beans 410g       |Groceries|16.99|600  |
|P005      |Nescafé Classic 200g       |Groceries|89.99|280  |
+----------+---------------------------+---------+-----+-----+
only showing top 5 rows



In [9]:
# Clean Orders
silver_orders = (
    df_orders
    .dropDuplicates(["order_id", "product_id"])          # One row per line item
    .withColumn("order_date",     F.to_date(F.col("order_date"), "yyyy-MM-dd"))
    .withColumn("quantity",       F.col("quantity").cast(IntegerType()))
    .withColumn("unit_price",     F.col("unit_price").cast(DoubleType()))
    .withColumn("line_total",     F.col("line_total").cast(DoubleType()))
    .withColumn("order_total",    F.col("order_total").cast(DoubleType()))
    .withColumn("store",          F.trim(F.col("store")))
    .withColumn("payment_method", F.trim(F.col("payment_method")))
    #recalculate line_total to guard against source errors
    .withColumn("line_total",     F.round(F.col("quantity") * F.col("unit_price"), 2))
    #add derived columns
    .withColumn("order_year",     F.year(F.col("order_date")))
    .withColumn("order_month",    F.month(F.col("order_date")))
    .withColumn("order_quarter",  F.quarter(F.col("order_date")))
    #filter out invalid rows
    .filter(F.col("quantity") > 0)
    .filter(F.col("unit_price") > 0)
    .dropna(subset=["order_id", "customer_id", "product_id", "order_date"])
    .select(
        "order_id", "customer_id", "order_date", "order_year",
        "order_month", "order_quarter", "store", "payment_method",
        "product_id", "quantity", "unit_price", "line_total", "order_total"
    )
)

print(f"\n silver_orders: {silver_orders.count()} rows")
silver_orders.show(5, truncate=False)

StatementMeta(, 831c6802-8da8-403f-a5e5-7569e6289849, 13, Finished, Available, Finished, False)


 silver_orders: 352 rows
+--------+-----------+----------+----------+-----------+-------------+-----------------------+--------------+----------+--------+----------+----------+-----------+
|order_id|customer_id|order_date|order_year|order_month|order_quarter|store                  |payment_method|product_id|quantity|unit_price|line_total|order_total|
+--------+-----------+----------+----------+-----------+-------------+-----------------------+--------------+----------+--------+----------+----------+-----------+
|ORD0001 |C004       |2025-08-13|2025      |8          |3            |Woolworths V&A         |Card          |P009      |2       |59.99     |119.98    |119.98     |
|ORD0002 |C018       |2025-05-07|2025      |5          |2            |Pick n Pay Sandton     |Card          |P019      |4       |14.99     |59.96     |59.96      |
|ORD0003 |C007       |2026-03-19|2026      |3          |1            |Pick n Pay Bloemfontein|Cash          |P001      |2       |19.99     |39.98     |264

In [11]:
# Data Quality Report
print("\n Data Quality Summary")
print("-" * 50)

for name, df in [("Customers", silver_customers),
 ("Products", silver_products), ("Orders", silver_orders)]:

 null_counts = {c: df.filter(F.col(c).isNull()).count() for c in df.columns}
 total_nulls = sum(null_counts.values())
 print(f"\n{name}: {df.count()} rows | {total_nulls} total nulls")
 if total_nulls > 0:
    for col, cnt in null_counts.items():
        if cnt > 0:
            print(f"{col}: {cnt} nulls")

StatementMeta(, 831c6802-8da8-403f-a5e5-7569e6289849, 15, Finished, Available, Finished, False)


 Data Quality Summary
--------------------------------------------------

Customers: 20 rows | 0 total nulls

Products: 20 rows | 0 total nulls

Orders: 352 rows | 0 total nulls


In [14]:
# Silver Delta Tables
print("\n Writing Silver Delta tables...")

# Customer
silver_customers.write.format("delta").mode("overwrite").saveAsTable("silver_customer")
print("silver_customer written")

# Products
silver_products.write.format("delta").mode("overwrite").saveAsTable("silver_product")
print("silver_product written")

# Orders
silver_orders.write.format("delta").mode("overwrite").saveAsTable("silver_orders")
print("silver_orders written")

print("Silver layer complete!")


StatementMeta(, 831c6802-8da8-403f-a5e5-7569e6289849, 18, Finished, Available, Finished, False)


 Writing Silver Delta tables...
silver_customer written
silver_product written
silver_orders written
Silver layer complete!
